# Step 1 pilot: prepare a shared LLM pilot set

Run this notebook after the Stockfish re-check. It does not rerun Step 1 or Stockfish; it makes one reproducible sample of five good and five bad checked positions.

Use the same `pilot_for_llm.csv` for every candidate LLM and keep each model's generations in a separate model-specific folder. The CSV deliberately omits good/bad labels, evaluations, and engine-best-move data. Keep `pilot_key.csv` private from the models.

In [ ]:
import datetime
import hashlib
import json
import platform
import random
import subprocess
import sys
from pathlib import Path

import pandas as pd

import config

ROOT = config.ROOT
MONTH = config.MONTH
SOURCE_DIR = config.OUT_DIR / "stockfish_recheck" / MONTH
STEP1_DIR = config.OUT_DIR / "step1_dataset" / MONTH
OUT = config.OUT_DIR / "step1_pilot" / MONTH
SOURCE_POSITIONS = SOURCE_DIR / "positions_stockfish.csv"
STOCKFISH_MANIFEST_PATH = SOURCE_DIR / "stockfish_manifest.json"
STEP1_MANIFEST_PATH = STEP1_DIR / "selection_manifest.json"
PILOT_FOR_LLM_PATH = OUT / "pilot_for_llm.csv"
PILOT_KEY_PATH = OUT / "pilot_key.csv"
PILOT_MANIFEST_PATH = OUT / "pilot_manifest.json"
PILOT_PER_KIND = config.PILOT_POSITIONS_PER_KIND
PILOT_SEED = config.SEED_PILOT_SPLIT

print(f"Month: {MONTH}; sample: {PILOT_PER_KIND} good + {PILOT_PER_KIND} bad")
print(f"Stockfish source: {SOURCE_POSITIONS}")
print(f"Pilot output: {OUT}")

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while chunk := stream.read(1 << 24):
            digest.update(chunk)
    return digest.hexdigest()


def hash_named_files(directory, names):
    digest = hashlib.sha256()
    for name in names:
        path = directory / name
        if not path.is_file():
            raise FileNotFoundError(f"Manifest output is missing: {path}")
        digest.update(name.encode("utf-8"))
        digest.update(path.read_bytes())
    return digest.hexdigest()


def tagged_notebook_sha256(stage):
    path = ROOT / "step1_select_positions.ipynb"
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source = "\n".join(
        "".join(cell["source"])
        for cell in notebook["cells"]
        if stage in cell.get("metadata", {}).get("tags", [])
    )
    return hashlib.sha256(source.encode("utf-8")).hexdigest()


def git(*args):
    try:
        result = subprocess.run(
            ["git", "-C", str(ROOT), *args],
            check=True,
            capture_output=True,
            text=True,
        )
        return result.stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


if not SOURCE_POSITIONS.is_file() or not STOCKFISH_MANIFEST_PATH.is_file():
    raise FileNotFoundError("Run the Stockfish re-check before creating the LLM pilot sample.")
if not STEP1_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"Step 1 manifest not found: {STEP1_MANIFEST_PATH}")

stockfish_manifest = json.loads(STOCKFISH_MANIFEST_PATH.read_text(encoding="utf-8"))
step1_manifest = json.loads(STEP1_MANIFEST_PATH.read_text(encoding="utf-8"))
if stockfish_manifest.get("month") != MONTH or step1_manifest.get("month") != MONTH:
    raise ValueError("The source manifests do not match config.MONTH.")
if step1_manifest.get("partial"):
    raise ValueError("Step 1 results are partial; complete Step 1 before sampling.")
if stockfish_manifest.get("step1_results_sha256") != step1_manifest.get("results_sha256"):
    raise ValueError("The Stockfish run references a different Step 1 result set.")
if stockfish_manifest.get("raw_sha256") != step1_manifest.get("raw_sha256"):
    raise ValueError("The Step 1 and Stockfish manifests reference different raw data.")

for name in ("STOCKFISH_VERSION", "STOCKFISH_DEPTH", "STOCKFISH_THREADS", "STOCKFISH_HASH_MB", "STOCKFISH_MULTIPV", "STOCKFISH_PATH"):
    if stockfish_manifest.get("config", {}).get(name) != str(getattr(config, name)):
        raise ValueError(f"Current config.{name} differs from the Stockfish run manifest.")

if tagged_notebook_sha256("step1") != step1_manifest.get("step1_source_sha256"):
    raise ValueError("Step 1 notebook source changed since its outputs were created.")
if tagged_notebook_sha256("step2") != stockfish_manifest.get("source_sha256"):
    raise ValueError("Step 2 notebook source changed since the Stockfish run.")

step1_input = STEP1_DIR / "positions_final.csv"
if sha256_file(step1_input) != stockfish_manifest.get("input_sha256"):
    raise ValueError("Step 1 input positions do not match the Stockfish run manifest.")
if hash_named_files(STEP1_DIR, step1_manifest.get("hashed_tables", [])) != step1_manifest.get("results_sha256"):
    raise ValueError("Step 1 output tables do not match selection_manifest.json.")
if "positions_stockfish.csv" not in stockfish_manifest.get("hashed_outputs", []):
    raise ValueError("Stockfish manifest does not include positions_stockfish.csv.")
if hash_named_files(SOURCE_DIR, stockfish_manifest["hashed_outputs"]) != stockfish_manifest.get("results_sha256"):
    raise ValueError("Stockfish output files do not match stockfish_manifest.json.")
print("Step 1 and Stockfish manifests, source code, and output hashes are consistent.")

In [ ]:
source_df = pd.read_csv(
    SOURCE_POSITIONS,
    dtype={"position_id": "string", "game_id": "string", "kind": "string"},
)
required_columns = {
    "position_id", "game_id", "kind", "fen", "side", "move_san",
    "stockfish_label", "stockfish_cp_loss", "stockfish_winprob_drop", "engine_best_san",
}
missing_columns = required_columns - set(source_df.columns)
if missing_columns:
    raise ValueError(f"Stockfish positions file lacks required columns: {sorted(missing_columns)}")
if source_df["position_id"].isna().any() or source_df["position_id"].duplicated().any():
    raise ValueError("Stockfish positions must have unique, non-empty position_id values.")
if not source_df["kind"].astype(str).equals(source_df["stockfish_label"].astype(str)):
    raise ValueError("Step 1 labels and Stockfish-confirmed labels differ in the source file.")
if len(source_df) != stockfish_manifest["kpis"]["kept_positions"]:
    raise ValueError("Stockfish positions row count differs from its manifest KPI.")

sampled_ids = {}
for kind in ("good", "bad"):
    available_ids = sorted(
        source_df.loc[source_df["kind"] == kind, "position_id"].astype(str).tolist()
    )
    if len(available_ids) < PILOT_PER_KIND:
        raise ValueError(f"Only {len(available_ids)} {kind} positions are available; need {PILOT_PER_KIND}.")
    sampler = random.Random(f"{PILOT_SEED}:step1-pilot:{kind}")
    sampled_ids[kind] = sampler.sample(available_ids, PILOT_PER_KIND)

selected_records = []
for kind in ("good", "bad"):
    selected_records.extend(
        source_df.loc[source_df["position_id"].isin(sampled_ids[kind])].to_dict(orient="records")
    )
random.Random(f"{PILOT_SEED}:step1-pilot:order").shuffle(selected_records)
selected = pd.DataFrame.from_records(selected_records)
selected.insert(0, "pilot_id", [f"P{index:03d}" for index in range(1, len(selected) + 1)])

pilot_for_llm = selected[["pilot_id", "fen", "side", "move_san"]].rename(
    columns={"side": "side_to_move", "move_san": "move_to_explain_san"}
)
pilot_key = selected[[
    "pilot_id", "position_id", "game_id", "kind", "stockfish_label",
    "stockfish_cp_loss", "stockfish_winprob_drop", "engine_best_san", "fen", "move_san",
]].copy()
if pilot_for_llm["pilot_id"].duplicated().any() or len(pilot_for_llm) != 2 * PILOT_PER_KIND:
    raise RuntimeError("The generated LLM pilot sample has invalid IDs or size.")
if selected["kind"].value_counts().to_dict() != {"good": PILOT_PER_KIND, "bad": PILOT_PER_KIND}:
    raise RuntimeError("The generated pilot sample is not balanced as configured.")

llm_bytes = pilot_for_llm.to_csv(index=False, lineterminator="\n").encode("utf-8")
key_bytes = pilot_key.to_csv(index=False, lineterminator="\n", float_format="%.10g").encode("utf-8")
output_bytes = {PILOT_FOR_LLM_PATH.name: llm_bytes, PILOT_KEY_PATH.name: key_bytes}
output_hash = hashlib.sha256()
for name in sorted(output_bytes):
    output_hash.update(name.encode("utf-8"))
    output_hash.update(output_bytes[name])

pilot_manifest = {
    "step": "step1_pilot_sample",
    "month": MONTH,
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "positions_per_kind": PILOT_PER_KIND,
    "seed": PILOT_SEED,
    "counts": {kind: int(count) for kind, count in selected["kind"].value_counts().items()},
    "source_stockfish_manifest_sha256": sha256_file(STOCKFISH_MANIFEST_PATH),
    "source_stockfish_results_sha256": stockfish_manifest["results_sha256"],
    "source_positions_sha256": sha256_file(SOURCE_POSITIONS),
    "source_step1_results_sha256": step1_manifest["results_sha256"],
    "source_step1_notebook_sha256": step1_manifest["step1_source_sha256"],
    "source_step2_notebook_sha256": stockfish_manifest["source_sha256"],
    "config": config.config_snapshot(),
    "llm_input_columns": list(pilot_for_llm.columns),
    "labels_in_llm_input": False,
    "llm_input_sha256": hashlib.sha256(llm_bytes).hexdigest(),
    "pilot_key_sha256": hashlib.sha256(key_bytes).hexdigest(),
    "hashed_outputs": sorted(output_bytes),
    "results_sha256": output_hash.hexdigest(),
    "git_commit": git("rev-parse", "HEAD"),
    "git_uncommitted_changes": bool(git("status", "--porcelain")),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "packages": {"pandas": pd.__version__},
}

if any(path.exists() for path in (PILOT_FOR_LLM_PATH, PILOT_KEY_PATH, PILOT_MANIFEST_PATH)):
    if not all(path.is_file() for path in (PILOT_FOR_LLM_PATH, PILOT_KEY_PATH, PILOT_MANIFEST_PATH)):
        raise RuntimeError("Pilot output is incomplete; preserve it before creating another sample.")
    existing_manifest = json.loads(PILOT_MANIFEST_PATH.read_text(encoding="utf-8"))
    if PILOT_FOR_LLM_PATH.read_bytes() != llm_bytes or PILOT_KEY_PATH.read_bytes() != key_bytes:
        raise RuntimeError("An existing pilot differs from this sample; it was not overwritten.")
    for key in ("source_stockfish_manifest_sha256", "seed", "positions_per_kind", "results_sha256"):
        if existing_manifest.get(key) != pilot_manifest[key]:
            raise RuntimeError(f"Existing pilot manifest differs for {key}; it was not overwritten.")
    print("The existing pilot sample is identical and was left unchanged.")
else:
    OUT.mkdir(parents=True, exist_ok=True)
    PILOT_FOR_LLM_PATH.write_bytes(llm_bytes)
    PILOT_KEY_PATH.write_bytes(key_bytes)
    PILOT_MANIFEST_PATH.write_text(json.dumps(pilot_manifest, indent=2) + "\n", encoding="utf-8")
    print("Created the fixed LLM pilot sample.")

print(f"Counts: {pilot_manifest['counts']}")
print(f"Give each model this same file: {PILOT_FOR_LLM_PATH}")
print(f"Keep this key out of model prompts: {PILOT_KEY_PATH}")
print(pilot_for_llm.to_string(index=False))